In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from torch.utils.data import random_split, DataLoader
from src.configs import SEED, BATCH_SIZE
from src.dataset import ImageDataset
from src.loss_function import Loss
from src.model import Model
from pathlib import Path
import torch

In [3]:
CLASS_TO_IDX = {
    "aeroplane": 0,
    "bicycle": 1,
    "bird": 2,
    "boat": 3,
    "bottle": 4,
    "bus": 5,
    "car": 6,
    "cat": 7,
    "chair": 8,
    "cow": 9,
    "diningtable": 10,
    "dog": 11,
    "horse": 12,
    "motorbike": 13,
    "person": 14,
    "pottedplant": 15,
    "sheep": 16,
    "sofa": 17,
    "train": 18,
    "tvmonitor": 19,
}

IDX_TO_CLASS = {
    idx: class_
    for class_, idx in CLASS_TO_IDX.items()
}

In [4]:
from torchvision.transforms import v2
import torch

# The mean and standard deviations across each channel for the normalized pixels
# of every single image in the "trainval" dataset
MEANS = (0.485, 0.456, 0.406)
STDS = (0.229, 0.224, 0.225)

trainval_transforms = v2.Compose([
    v2.Normalize(mean=MEANS, std=STDS)
])

In [39]:
annot_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annot_file_trainval, img_dir_trainval,
                                transform=trainval_transforms)

generator_ = torch.Generator().manual_seed(SEED)
train_dataset, _, val_dataset = random_split(trainval_dataset, [0.3, 0.4, 0.3]
                                          ,generator=generator_)

len(train_dataset), len(val_dataset)

(1504, 1503)

In [40]:
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=4, pin_memory=True, persistent_workers=True)

In [41]:
val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=4, pin_memory=True, persistent_workers=True)

In [42]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [43]:
model = Model().to(device, non_blocking=True)
loss_fn = Loss().to(device, non_blocking=True)

optimizer = torch.optim.Adam([
    {"params": model.backbone.parameters(), "lr": 1e-4},
    {"params": model.detector_head.parameters()}
    ], lr=1e-3)

In [44]:
num_epochs=60

In [45]:
from torch.optim.lr_scheduler import CosineAnnealingLR 

scheduler = CosineAnnealingLR(optimizer, num_epochs, 1e-6)

In [46]:
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_dl:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)
        loss = loss_fn(preds, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()

    print(epoch, epoch_loss / len(train_dl))

0 31.70690924056033
1 8.486686280433167
2 7.293832271657092
3 8.183674944208024
4 6.33506182406811
5 5.884256393351453
6 5.622240832511415
7 5.349133552388942
8 5.095821867597864
9 4.786108874260111
10 4.546989324245047
11 4.274741451790992
12 4.013712522831369
13 3.691760301589966
14 3.4979141671606837
15 3.200840655793535
16 2.9306348232512778
17 2.7309968167162957
18 2.6162275202730867
19 2.3912679971532618
20 2.193068374978735
21 2.030871718487841
22 1.8940717793525534
23 1.7616816784473175
24 1.603329653435565
25 1.5317295142944822
26 1.3946011801983447
27 1.2953953590798886
28 1.1529050228443551
29 1.0972637090277164
30 0.9916343245100467
31 0.9519746164058117
32 0.921420990152562
33 0.8438832309651882
34 0.7942639817582801
35 0.7579431051903582
36 0.7152863106829055
37 0.6768857018744692
38 0.6687154142146415
39 0.6286526867683898
40 0.6299437880516052
41 0.5875462217533842
42 0.5683874189853668
43 0.5422191315508903
44 0.533720334793659
45 0.5243128519108955
46 0.51264649756411

KeyboardInterrupt: 

In [47]:
from src.inference_functions import compute_eval_stats

train_mAP = compute_eval_stats(model, train_dl, device, test=True)
train_mAP

(0.908491032453638,
 {'aeroplane': 0.9565282564371329,
  'bicycle': 0.9105608441949936,
  'bird': 0.8824712119201379,
  'boat': 0.8612324481220479,
  'bottle': 0.7088074206431842,
  'bus': 0.9374622297476495,
  'car': 0.9015315785737816,
  'cat': 0.9821428571428571,
  'chair': 0.9123033060509728,
  'cow': 0.9262462960324978,
  'diningtable': 0.8715405194235585,
  'dog': 0.9606299212598425,
  'horse': 0.9197248846333488,
  'motorbike': 0.918918918918919,
  'person': 0.8729387543082568,
  'pottedplant': 0.883886585722647,
  'sheep': 0.9230655500498335,
  'sofa': 0.9700832193675236,
  'train': 0.9580456631848835,
  'tvmonitor': 0.9117001833386933},
 {'aeroplane': ([1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,


In [48]:
val_mAP = compute_eval_stats(model, val_dl, device, test=True)
val_mAP

(0.05121166167770197,
 {'aeroplane': 0.0886958662889514,
  'bicycle': 0.09199448409977443,
  'bird': 0.027164291291763443,
  'boat': 0.0013814994731025266,
  'bottle': 0.0023148148148148147,
  'bus': 0.08394015622931286,
  'car': 0.07629614467408755,
  'cat': 0.09564772817761948,
  'chair': 0.006257050448136741,
  'cow': 0.03594754078396177,
  'diningtable': 0.02109704641350211,
  'dog': 0.030342784201370977,
  'horse': 0.12006370659765768,
  'motorbike': 0.09910812160462834,
  'person': 0.0808558909828863,
  'pottedplant': 0.005389112166605772,
  'sheep': 0.008792250797259323,
  'sofa': 0.0464196354104611,
  'train': 0.03438949938949939,
  'tvmonitor': 0.06813560970864341},
 {'aeroplane': ([1.0,
    1.0,
    0.6666666666666666,
    0.75,
    0.6,
    0.6666666666666666,
    0.5714285714285714,
    0.5,
    0.5555555555555556,
    0.6,
    0.5454545454545454,
    0.5833333333333334,
    0.6153846153846154,
    0.5714285714285714,
    0.5333333333333333,
    0.5,
    0.47058823529411764

In [12]:
X_batch, y_batch = next(iter(train_dl))
X_batch = X_batch.to(device, non_blocking=True)

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 25]))

In [13]:
preds = model(X_batch)
preds.shape

torch.Size([32, 7, 7, 25])

In [14]:
item_row_col = []
classes = []

for item in range(5):
    for i in range(7):
        for j in range(7):
            if y_batch[item][i][j][-1] == 1:
                item_row_col.append((item, i, j))
                classes.append(y_batch[item][i][j][:20].argmax())

In [15]:
item_row_col, classes

([(0, 2, 4),
  (0, 3, 2),
  (1, 4, 1),
  (1, 4, 2),
  (1, 4, 6),
  (1, 5, 0),
  (2, 2, 1),
  (2, 2, 2),
  (2, 3, 0),
  (2, 3, 1),
  (2, 3, 2),
  (2, 3, 4),
  (3, 3, 4),
  (4, 2, 3),
  (4, 4, 3)],
 [tensor(2),
  tensor(2),
  tensor(8),
  tensor(15),
  tensor(14),
  tensor(15),
  tensor(18),
  tensor(18),
  tensor(14),
  tensor(14),
  tensor(14),
  tensor(18),
  tensor(7),
  tensor(14),
  tensor(12)])

In [16]:
IDX_TO_CLASS[14], IDX_TO_CLASS[19], IDX_TO_CLASS[7], IDX_TO_CLASS[18], IDX_TO_CLASS[5], IDX_TO_CLASS[9]

('person', 'tvmonitor', 'cat', 'train', 'bus', 'cow')

In [17]:
preds[2][0][1], y_batch[2][0][1]

(tensor([ 0.4680,  0.0448,  0.1434, -0.1429, -0.5060, -0.3011,  0.8231,  0.4585,
          0.1217, -0.6924,  0.1563,  0.3304, -0.9735, -0.3996,  1.0403,  0.5771,
          0.2990, -0.6772,  0.5346,  0.4610,  0.5714,  0.8817,  1.2778,  0.9909,
          0.0013], device='cuda:0', grad_fn=<SelectBackward0>),
 tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0.]))

In [18]:
target_class = y_batch[2][1][6][:20].argmax()
pred_class = preds[2][1][6][:20].argmax()

target_class, pred_class, preds[2][1][6][target_class]

(tensor(0),
 tensor(14, device='cuda:0'),
 tensor(-0.3115, device='cuda:0', grad_fn=<SelectBackward0>))

In [19]:
from src.utilities import IoU, convert_xywh_coordinates

bbox_1 = preds[2][1][6][20:24]
bbox_2 = y_batch[2][1][6][20:24]

bbox_1 = convert_xywh_coordinates(bbox_1, 1, 6, draw=False)
bbox_2 = convert_xywh_coordinates(bbox_2, 1, 6, draw=False)

IoU(bbox_1, bbox_2)

tensor(0., device='cuda:0', grad_fn=<DivBackward0>)